# microWakeWord — Train any wake word

A single-notebook trainer for custom wake words on ESPHome `micro_wake_word` devices
(M5Stack Atom Echo, Voice PE, etc).

Two modes:
- **`generate`** — Piper TTS generates ~30k positive samples + your confusables in Colab.
  Easiest path. Requires an IPA pronunciation of your wake word.
- **`bundle`** — You upload a zip with your own samples (real recordings, ElevenLabs
  voices, accent-matched TTS). Higher quality but you do the prep work.

## Required runtime
**Runtime → Change runtime type → A100 GPU + High-RAM**.
T4 OOMs during validation. A100 + High-RAM gives 40 GB VRAM + 85 GB system RAM.

## What you get
A `<output_name>.tflite` (~60 KB) + companion `.json` manifest, ready to drop into
your ESPHome config under `micro_wake_word: models:`.

## Workflow
1. Edit the **CONFIGURE HERE** cell below for your wake word
2. **Runtime → Run all**, walk away ~45 minutes
3. Find `<output_name>.tflite` + `.json` in your Drive folder when it's done
4. Test on hardware — likely needs manifest tuning (cutoff, sliding_window) for
   your specific model. See the deployment notes at the end.

Built from working production deployment of "Hey Harold" — all known
upstream bugs are patched in this notebook.


In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════╗
# ║                    CONFIGURE YOUR WAKE WORD HERE                       ║
# ╚═══════════════════════════════════════════════════════════════════════╝

# ─── Wake word identity ───
WAKE_WORD = "Hey Harold"           # Human-readable name (shown in HA)
OUTPUT_NAME = "hey_harold"         # Filename (no spaces, lowercase)
AUTHOR = "your_name"               # Goes in the manifest
AUTHOR_WEBSITE = "https://github.com/yourname"

# ─── Mode ───
MODE = "generate"   # "generate" (Piper makes samples) | "bundle" (you uploaded a zip)

# ─── Drive folder (created if missing) ───
DRIVE_FOLDER = "wakeword_training_hey_harold"

# ─── If MODE == "generate" ───
# Look up IPA for your wake word: https://www.internationalphoneticassociation.org/IPA-chart
# or use eSpeak: `espeak-ng -q --ipa "Hey Harold"` on Linux
WAKE_WORD_IPA_US = "hˈeɪ hˈærəld"   # US English pronunciation
WAKE_WORD_IPA_UK = "hˈeɪ hˈɛrəld"   # UK English (set to None to skip second pass)
SAMPLES_US = 30000                  # ~12 min on T4, ~7 min on A100
SAMPLES_UK = 15000                  # 0 to skip UK pass

# Confusable phrases — should NOT trigger your wake word.
# Include: phonetic neighbors, the bare name without prefix, common
# false-trigger phrases like other assistant names.
CONFUSABLE_PHRASES = [
    # General "hey X" near-misses
    "hey there", "hey you", "hey y'all", "hey now",
    # Other assistant wake words (must not steal yours)
    "hey siri", "hey google", "okay google", "hey alexa", "okay nabu",
    # WAKE-WORD SPECIFIC — replace these for your word!
    "hey howard", "hey harvey", "hey gerald", "hey carol",  # H-name neighbors
    "harold", "the herald",                                  # bare name + similar
    "hairy old", "hello harold",                              # phonetic mash-ups
]
SAMPLES_PER_CONFUSABLE = 1000   # 500 = light, 1000 = recommended, 2000 = max

# ─── If MODE == "bundle" ───
# Expected: <DRIVE_FOLDER>/data_bundle.zip with this layout:
#   generated_samples/        positive samples (.wav, 16 kHz mono)
#   real_recordings/          (optional) real mic recordings
#   confusable_negatives/     (optional) hard negative samples
BUNDLE_NAME = "data_bundle.zip"

# ─── Manifest tuning (works well as defaults; tune after on-device testing) ───
PROBABILITY_CUTOFF = 0.85          # 0.5 = too lenient, 0.97 = okay_nabu strict
SLIDING_WINDOW_SIZE = 5            # 5 = okay_nabu default; higher = slower fire
TENSOR_ARENA_SIZE = 50000          # 30000 works, 50000 has more headroom

# ─── Languages this model is trained for (manifest metadata) ───
TRAINED_LANGUAGES = ["en"]         # add "it", "es" etc if you have multi-accent samples

# ╔═══════════════════════════════════════════════════════════════════════╗
# ║                           END OF CONFIG                                ║
# ╚═══════════════════════════════════════════════════════════════════════╝
print(f"Training '{WAKE_WORD}' as {OUTPUT_NAME} in mode={MODE!r}")
print(f"Output → /content/drive/MyDrive/{DRIVE_FOLDER}/{OUTPUT_NAME}.tflite")


In [2]:
# === MIRA V2 — RECOVER PIPER AFTER COLAB RUNTIME RESET ===

import os
import sys
import json
import shutil
import subprocess
from pathlib import Path

os.chdir("/content")

# ------------------------------------------------------------
# 1. Install required packages
# ------------------------------------------------------------

print("Installing Piper requirements...")

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade",
        "piper-tts==1.3.0",
        "piper-sample-generator",
        "cython",
        "setuptools",
        "wheel",
    ],
    check=True,
)

# ------------------------------------------------------------
# 2. Clone legacy Piper source used by the .pt generator
# ------------------------------------------------------------

piper_dir = Path("/content/piper")

if not piper_dir.exists():
    print("Cloning Piper source...")

    subprocess.run(
        [
            "git",
            "clone",
            "-q",
            "https://github.com/rhasspy/piper.git",
            str(piper_dir),
        ],
        check=True,
    )

# ------------------------------------------------------------
# 3. Rebuild monotonic_align
# ------------------------------------------------------------

align_dir = Path(
    "/content/piper/src/python/"
    "piper_train/vits/monotonic_align"
)

target_dir = align_dir / "monotonic_align"
target_dir.mkdir(exist_ok=True)

existing_so = list(target_dir.glob("core*.so"))

if not existing_so:
    print("Building monotonic_align...")

    subprocess.run(
        [
            "cythonize",
            "-i",
            "core.pyx",
        ],
        cwd=str(align_dir),
        check=True,
    )

    compiled = list(align_dir.glob("core*.so"))

    if not compiled:
        raise RuntimeError(
            "monotonic_align compiled but no .so was found."
        )

    for so_file in compiled:
        destination = target_dir / so_file.name

        if destination.exists():
            destination.unlink()

        shutil.move(
            str(so_file),
            str(destination),
        )

print("monotonic_align ready.")

# ------------------------------------------------------------
# 4. Set Piper Python path
# ------------------------------------------------------------

os.environ["PYTHONPATH"] = ":".join([
    "/content/piper/src/python",
    os.environ.get("PYTHONPATH", ""),
])

# ------------------------------------------------------------
# 5. Download LibriTTS-R generator model
# ------------------------------------------------------------

models_dir = Path("/content/models")
models_dir.mkdir(parents=True, exist_ok=True)

model_path = (
    models_dir /
    "en_US-libritts_r-medium.pt"
)

if not model_path.exists():
    print("Downloading LibriTTS-R model...")

    subprocess.run(
        [
            "wget",
            "-q",
            "--show-progress",
            "-O",
            str(model_path),
            (
                "https://github.com/rhasspy/"
                "piper-sample-generator/releases/download/"
                "v2.0.0/en_US-libritts_r-medium.pt"
            ),
        ],
        check=True,
    )

# ------------------------------------------------------------
# 6. Download FULL official Piper configuration
# ------------------------------------------------------------

config_path = Path(
    str(model_path) + ".json"
)

print("Downloading full LibriTTS-R config...")

subprocess.run(
    [
        "wget",
        "-q",
        "-O",
        str(config_path),
        (
            "https://huggingface.co/rhasspy/"
            "piper-voices/resolve/main/"
            "en/en_US/libritts_r/medium/"
            "en_US-libritts_r-medium.onnx.json"
        ),
    ],
    check=True,
)

# Validate config
with config_path.open(
    "r",
    encoding="utf-8"
) as f:
    config = json.load(f)

required = [
    "audio",
    "espeak",
    "phoneme_id_map",
    "num_speakers",
]

missing = [
    key
    for key in required
    if key not in config
]

if missing:
    raise RuntimeError(
        f"Config missing fields: {missing}"
    )

# ------------------------------------------------------------
# 7. Verify everything
# ------------------------------------------------------------

test = subprocess.run(
    [
        sys.executable,
        "-c",
        (
            "import piper_sample_generator; "
            "from piper_train.vits.monotonic_align "
            "import maximum_path; "
            "print('PIPER_IMPORTS_OK')"
        ),
    ],
    env=os.environ.copy(),
    capture_output=True,
    text=True,
)

print()
print(test.stdout)

if test.returncode != 0:
    print(test.stderr)
    raise RuntimeError(
        "Piper import verification failed."
    )

print("======================================")
print("MIRA V2 PIPER RECOVERY COMPLETE")
print("======================================")
print(
    "MODEL:",
    model_path,
    f"{model_path.stat().st_size / 1024 / 1024:.1f} MB"
)
print(
    "CONFIG:",
    config_path,
    f"{config_path.stat().st_size / 1024:.1f} KB"
)
print(
    "Sample rate:",
    config["audio"]["sample_rate"]
)
print(
    "Speakers:",
    config["num_speakers"]
)
print(
    "Phoneme entries:",
    len(config["phoneme_id_map"])
)

Installing Piper requirements...
Cloning Piper source...
Building monotonic_align...
monotonic_align ready.

PIPER_IMPORTS_OK

MIRA V2 PIPER RECOVERY COMPLETE
MODEL: /content/models/en_US-libritts_r-medium.pt 194.6 MB
CONFIG: /content/models/en_US-libritts_r-medium.pt.json 19.7 KB
Sample rate: 22050
Speakers: 904
Phoneme entries: 159


In [3]:
# === MIRA V2 — 10-SAMPLE GENERATION TEST ===

import os
import sys
import shutil
import subprocess
from pathlib import Path

os.chdir("/content")

model_path = Path(
    "/content/models/en_US-libritts_r-medium.pt"
)

test_dir = Path(
    "/content/mira_v2_test"
)

env = os.environ.copy()

env["PYTHONPATH"] = ":".join([
    "/content/piper/src/python",
    env.get("PYTHONPATH", ""),
])

if test_dir.exists():
    shutil.rmtree(test_dir)

test_dir.mkdir(
    parents=True,
    exist_ok=True
)

cmd = [
    sys.executable,
    "-m",
    "piper_sample_generator",
    "Mira",
    "--model",
    str(model_path),
    "--max-samples",
    "10",
    "--batch-size",
    "8",
    "--max-speakers",
    "50",
    "--length-scales",
    "0.9",
    "1.0",
    "1.1",
    "--output-dir",
    str(test_dir),
]

print("======================================")
print("RUNNING 10-SAMPLE MIRA TEST")
print("======================================")
print()

proc = subprocess.Popen(
    cmd,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in proc.stdout:
    print(line, end="")

proc.wait()

wavs = sorted(
    test_dir.glob("*.wav")
)

print()
print("Exit code:", proc.returncode)
print("TEST WAV FILES:", len(wavs))

if proc.returncode != 0:
    raise RuntimeError(
        "Mira 10-sample generation test failed."
    )

if len(wavs) != 10:
    raise RuntimeError(
        f"Expected 10 WAV files, found {len(wavs)}."
    )

print()
print("FIRST:", wavs[0])
print("LAST :", wavs[-1])

print()
print("======================================")
print("MIRA 10-SAMPLE TEST PASSED")
print("======================================")

RUNNING 10-SAMPLE MIRA TEST

DEBUG:__main__:Loading /content/models/en_US-libritts_r-medium.pt
INFO:__main__:Successfully loaded the model
DEBUG:__main__:CUDA available, using GPU
DEBUG:__main__:Batch 1/1 complete
DEBUG:__main__:Batch 2/1 complete
INFO:__main__:Done

Exit code: 0
TEST WAV FILES: 10

FIRST: /content/mira_v2_test/0.wav
LAST : /content/mira_v2_test/9.wav

MIRA 10-SAMPLE TEST PASSED


In [4]:
# === MIRA V2 — FULL SYNTHETIC DATA GENERATION ===

import os
import sys
import shutil
import subprocess
import tarfile
from pathlib import Path

from google.colab import drive

# ------------------------------------------------------------
# Mount Drive
# ------------------------------------------------------------

drive.mount("/content/drive")

os.chdir("/content")

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

model_path = Path(
    "/content/models/en_US-libritts_r-medium.pt"
)

drive_root = Path(
    "/content/drive/MyDrive/wakeword_training_mira_v2"
)

real_voice_dir = drive_root / "real_voice"
backup_dir = drive_root / "generated_backups"

positive_dir = Path(
    "/content/generated_samples_v2"
)

confusable_root = Path(
    "/content/confusable_negatives_v2"
)

flat_confusables = Path(
    "/content/confusable_negatives_v2_flat"
)

backup_dir.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# Verify model + real recordings
# ------------------------------------------------------------

if not model_path.exists():
    raise RuntimeError(
        f"Generator model missing: {model_path}"
    )

real_wavs = sorted(
    real_voice_dir.glob("*.wav")
)

print("======================================")
print("MIRA V2 DATA PREFLIGHT")
print("======================================")
print("Real voice samples:", len(real_wavs))
print("Generator model:", model_path)
print("Drive backup:", backup_dir)

if len(real_wavs) != 52:
    raise RuntimeError(
        f"Expected 52 real voice recordings, found {len(real_wavs)}"
    )

# ------------------------------------------------------------
# Piper environment
# ------------------------------------------------------------

env = os.environ.copy()

env["PYTHONPATH"] = ":".join([
    "/content/piper/src/python",
    env.get("PYTHONPATH", ""),
])

# ------------------------------------------------------------
# Generator helper
# ------------------------------------------------------------

def generate_phrase(
    phrase,
    count,
    output_dir,
    max_speakers=400,
    batch_size=32,
):

    output_dir = Path(output_dir)

    existing = sorted(
        output_dir.glob("*.wav")
    ) if output_dir.exists() else []

    # Skip an already-complete set
    if len(existing) == count:
        print()
        print(
            f'SKIPPING "{phrase}" — '
            f'{count} samples already exist.'
        )
        return

    # Remove partial/failed run
    if output_dir.exists():
        shutil.rmtree(output_dir)

    output_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    cmd = [
        sys.executable,
        "-m",
        "piper_sample_generator",
        phrase,
        "--model",
        str(model_path),
        "--max-samples",
        str(count),
        "--batch-size",
        str(batch_size),
        "--max-speakers",
        str(max_speakers),
        "--length-scales",
        "0.85",
        "0.95",
        "1.0",
        "1.1",
        "1.2",
        "--output-dir",
        str(output_dir),
    ]

    print()
    print("=" * 60)
    print(f'GENERATING: "{phrase}"')
    print("Samples:", count)
    print("Output :", output_dir)
    print("=" * 60)

    proc = subprocess.Popen(
        cmd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    for line in proc.stdout:
        print(line, end="")

    proc.wait()

    wavs = sorted(
        output_dir.glob("*.wav")
    )

    print()
    print("Exit code:", proc.returncode)
    print("WAV files:", len(wavs))

    if proc.returncode != 0:
        raise RuntimeError(
            f'Generation failed for "{phrase}"'
        )

    if len(wavs) != count:
        raise RuntimeError(
            f'Expected {count} WAVs for "{phrase}", '
            f'found {len(wavs)}'
        )


def make_backup(
    source_dir,
    archive_path,
):

    source_dir = Path(source_dir)
    archive_path = Path(archive_path)

    print()
    print(
        "Backing up:",
        source_dir.name
    )

    with tarfile.open(
        archive_path,
        "w:gz"
    ) as tar:

        tar.add(
            source_dir,
            arcname=source_dir.name
        )

    print(
        "Saved:",
        archive_path,
        f"({archive_path.stat().st_size / 1024 / 1024:.1f} MB)"
    )

# ------------------------------------------------------------
# 1. Generate 12,000 Mira positives
# ------------------------------------------------------------

generate_phrase(
    "Mira",
    12000,
    positive_dir,
    max_speakers=400,
    batch_size=32,
)

positive_backup = (
    backup_dir /
    "synthetic_mira_12000.tar.gz"
)

make_backup(
    positive_dir,
    positive_backup
)

# ------------------------------------------------------------
# 2. Generate confusable negatives
# ------------------------------------------------------------

CONFUSABLE_PHRASES = [
    "Mia",
    "Mila",
    "Mina",
    "Myra",
    "Kira",
    "Vera",
    "mirror",
    "Miranda",
    "miracle",
    "nearer",
    "Siri",
    "Alexa",
    "Gemini",
    "hey Google",
    "okay Google",
    "okay Nabu",
]

confusable_root.mkdir(
    parents=True,
    exist_ok=True
)

for number, phrase in enumerate(
    CONFUSABLE_PHRASES,
    start=1
):

    safe_name = (
        phrase.lower()
        .replace(" ", "_")
        .replace("-", "_")
    )

    phrase_dir = (
        confusable_root /
        safe_name
    )

    print()
    print(
        f"CONFUSABLE {number}/"
        f"{len(CONFUSABLE_PHRASES)}"
    )

    generate_phrase(
        phrase,
        400,
        phrase_dir,
        max_speakers=400,
        batch_size=32,
    )

# ------------------------------------------------------------
# 3. Flatten confusables with unique names
# ------------------------------------------------------------

print()
print("Flattening confusable samples...")

if flat_confusables.exists():
    shutil.rmtree(
        flat_confusables
    )

flat_confusables.mkdir(
    parents=True,
    exist_ok=True
)

counter = 0

for phrase_dir in sorted(
    confusable_root.iterdir()
):

    if not phrase_dir.is_dir():
        continue

    safe_name = phrase_dir.name

    for wav in sorted(
        phrase_dir.glob("*.wav")
    ):

        counter += 1

        destination = (
            flat_confusables /
            f"{safe_name}_{counter:05d}.wav"
        )

        shutil.copy2(
            wav,
            destination
        )

# ------------------------------------------------------------
# 4. Final verification
# ------------------------------------------------------------

positive_count = len(
    list(
        positive_dir.glob("*.wav")
    )
)

negative_count = len(
    list(
        flat_confusables.glob("*.wav")
    )
)

print()
print("======================================")
print("MIRA V2 DATASET COUNTS")
print("======================================")
print(
    "Synthetic Mira positives:",
    positive_count
)
print(
    "Confusable negatives:    ",
    negative_count
)
print(
    "Real Ali Mira recordings:",
    len(real_wavs)
)

if positive_count != 12000:
    raise RuntimeError(
        f"Expected 12000 positives, got {positive_count}"
    )

if negative_count != 6400:
    raise RuntimeError(
        f"Expected 6400 confusables, got {negative_count}"
    )

# ------------------------------------------------------------
# 5. Back up flattened confusables
# ------------------------------------------------------------

negative_backup = (
    backup_dir /
    "confusable_negatives_6400.tar.gz"
)

make_backup(
    flat_confusables,
    negative_backup
)

print()
print("======================================")
print("MIRA V2 SYNTHETIC DATA COMPLETE")
print("======================================")
print("Positive Mira samples:", positive_count)
print("Confusable samples:   ", negative_count)
print("Real voice samples:   ", len(real_wavs))
print()
print("BACKUPS:")
print(positive_backup)
print(negative_backup)

Mounted at /content/drive
MIRA V2 DATA PREFLIGHT
Real voice samples: 52
Generator model: /content/models/en_US-libritts_r-medium.pt
Drive backup: /content/drive/MyDrive/wakeword_training_mira_v2/generated_backups

GENERATING: "Mira"
Samples: 12000
Output : /content/generated_samples_v2
DEBUG:__main__:Loading /content/models/en_US-libritts_r-medium.pt
INFO:__main__:Successfully loaded the model
DEBUG:__main__:CUDA available, using GPU
DEBUG:__main__:Batch 1/375 complete
DEBUG:__main__:Batch 2/375 complete
DEBUG:__main__:Batch 3/375 complete
DEBUG:__main__:Batch 4/375 complete
DEBUG:__main__:Batch 5/375 complete
DEBUG:__main__:Batch 6/375 complete
DEBUG:__main__:Batch 7/375 complete
DEBUG:__main__:Batch 8/375 complete
DEBUG:__main__:Batch 9/375 complete
DEBUG:__main__:Batch 10/375 complete
DEBUG:__main__:Batch 11/375 complete
DEBUG:__main__:Batch 12/375 complete
DEBUG:__main__:Batch 13/375 complete
DEBUG:__main__:Batch 14/375 complete
DEBUG:__main__:Batch 15/375 complete
DEBUG:__main__:B

In [5]:
# === MIRA V2 — PRONUNCIATION SANITY CHECK ===
# Listen to several synthetic training samples before feature extraction.

from pathlib import Path
from IPython.display import Audio, display
import random

synthetic_dir = Path("/content/generated_samples_v2")

wavs = sorted(synthetic_dir.glob("*.wav"))

print("Synthetic Mira files:", len(wavs))

if len(wavs) != 12000:
    raise RuntimeError(
        f"Expected 12000 synthetic samples, found {len(wavs)}"
    )

random.seed(42)
chosen = random.sample(wavs, 6)

print()
print("Listen to these six.")
print('They should all sound like "MEE-rah".')
print()

for i, wav in enumerate(chosen, 1):
    print(f"Sample {i}: {wav.name}")
    display(Audio(str(wav)))

Synthetic Mira files: 12000

Listen to these six.
They should all sound like "MEE-rah".

Sample 1: 8627.wav


Sample 2: 11639.wav


Sample 3: 10365.wav


Sample 4: 3253.wav


Sample 5: 2809.wav


Sample 6: 249.wav


In [8]:
# === MIRA V2 — FINISH AUDIOMENTATIONS DEPENDENCIES ===

import sys
import subprocess
import importlib

print("Installing missing audiomentations dependencies...")

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "numpy-minmax>=0.3.0,<1",
        "numpy-rms>=0.4.2,<1",
        "python-stretch>=0.3.1,<1",
        "soxr>=0.3.2,<1.0.0",
    ],
    check=True,
)

# Clear any half-loaded modules from the previous failed import
for name in list(sys.modules):
    if (
        name == "audiomentations"
        or name.startswith("audiomentations.")
        or name == "microwakeword.audio.augmentation"
    ):
        del sys.modules[name]

importlib.invalidate_caches()

# ------------------------------------------------------------
# Verify audiomentations
# ------------------------------------------------------------

import audiomentations

print()
print("=== AUDIOMENTATIONS ===")
print(
    "Version:",
    getattr(audiomentations, "__version__", "unknown")
)
print(
    "AddColorNoise:",
    hasattr(audiomentations, "AddColorNoise")
)
print(
    "PitchShift:",
    hasattr(audiomentations, "PitchShift")
)
print(
    "BandStopFilter:",
    hasattr(audiomentations, "BandStopFilter")
)

if not hasattr(audiomentations, "AddColorNoise"):
    raise RuntimeError(
        "AddColorNoise is still unavailable."
    )

# ------------------------------------------------------------
# Verify the exact microWakeWord augmenter we need
# ------------------------------------------------------------

from microwakeword.audio.augmentation import Augmentation

test_augmenter = Augmentation(
    augmentation_duration_s=3.2,

    augmentation_probabilities={
        "SevenBandParametricEQ": 0.15,
        "TanhDistortion": 0.10,
        "PitchShift": 0.15,
        "BandStopFilter": 0.10,
        "AddColorNoise": 0.30,
        "AddBackgroundNoise": 0.0,
        "RIR": 0.0,
        "Gain": 1.0,
        "GainTransition": 0.25,
    },

    impulse_paths=[],
    background_paths=[],

    background_min_snr_db=-5,
    background_max_snr_db=20,

    min_jitter_s=0.10,
    max_jitter_s=0.50,
)

print()
print("======================================")
print("AUDIOMENTATIONS ENVIRONMENT READY")
print("======================================")

Installing missing audiomentations dependencies...

=== AUDIOMENTATIONS ===
Version: 0.43.1
AddColorNoise: True
PitchShift: True
BandStopFilter: True

AUDIOMENTATIONS ENVIRONMENT READY


In [9]:
# === MIRA V2 — PERSONALIZED FEATURE EXTRACTION ===
# Uses:
#   12,000 synthetic Mira positives
#   6,400 confusable negatives
#   52 real Ali "Mira" recordings
#
# Real voice gets 25x training repetition.
# No external ambient/RIR datasets in this quick v2 pass.

import os
import shutil
import subprocess
from pathlib import Path

from google.colab import drive

os.chdir("/content")

# ------------------------------------------------------------
# 1. Make sure Drive is available
# ------------------------------------------------------------

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

drive_root = Path(
    "/content/drive/MyDrive/wakeword_training_mira_v2"
)

backup_root = drive_root / "generated_backups"

# ------------------------------------------------------------
# 2. Input paths
# ------------------------------------------------------------

synthetic_dir = Path(
    "/content/generated_samples_v2"
)

confusable_dir = Path(
    "/content/confusable_negatives_v2_flat"
)

real_dir = Path(
    "/content/real_recordings"
)

real_drive_dir = (
    drive_root / "real_voice"
)

# ------------------------------------------------------------
# 3. Recover raw datasets if necessary
# ------------------------------------------------------------

def wav_count(path):
    path = Path(path)

    if not path.exists():
        return 0

    return len(list(path.glob("*.wav")))


# Synthetic
if wav_count(synthetic_dir) != 12000:
    print("Recovering synthetic Mira samples...")

    if synthetic_dir.exists():
        shutil.rmtree(synthetic_dir)

    archive = (
        backup_root /
        "synthetic_mira_12000.tar.gz"
    )

    if not archive.exists():
        raise RuntimeError(
            f"Missing backup: {archive}"
        )

    subprocess.run(
        [
            "tar",
            "-xzf",
            str(archive),
            "-C",
            "/content",
        ],
        check=True,
    )


# Confusables
if wav_count(confusable_dir) != 6400:
    print("Recovering confusable samples...")

    if confusable_dir.exists():
        shutil.rmtree(confusable_dir)

    archive = (
        backup_root /
        "confusable_negatives_6400.tar.gz"
    )

    if not archive.exists():
        raise RuntimeError(
            f"Missing backup: {archive}"
        )

    subprocess.run(
        [
            "tar",
            "-xzf",
            str(archive),
            "-C",
            "/content",
        ],
        check=True,
    )


# Real voice
if wav_count(real_dir) != 52:
    print("Recovering real Mira recordings...")

    if real_dir.exists():
        shutil.rmtree(real_dir)

    if not real_drive_dir.exists():
        raise RuntimeError(
            f"Missing real voice folder: {real_drive_dir}"
        )

    shutil.copytree(
        real_drive_dir,
        real_dir,
    )

# ------------------------------------------------------------
# 4. Verify all input data
# ------------------------------------------------------------

synthetic_count = wav_count(synthetic_dir)
confusable_count = wav_count(confusable_dir)
real_count = wav_count(real_dir)

print()
print("======================================")
print("MIRA V2 FEATURE INPUTS")
print("======================================")
print("Synthetic positives:", synthetic_count)
print("Confusable negatives:", confusable_count)
print("Real Ali recordings:", real_count)

assert synthetic_count == 12000
assert confusable_count == 6400
assert real_count == 52

# ------------------------------------------------------------
# 5. Verify our corrected audiomentations install
# ------------------------------------------------------------

import audiomentations

print()
print(
    "Audiomentations:",
    getattr(
        audiomentations,
        "__version__",
        "unknown",
    ),
)

if not hasattr(
    audiomentations,
    "AddColorNoise",
):
    raise RuntimeError(
        "AddColorNoise disappeared. "
        "Do not continue with feature extraction."
    )

# ------------------------------------------------------------
# 6. microWakeWord imports
# ------------------------------------------------------------

from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration
from mmap_ninja.ragged import RaggedMmap

# ------------------------------------------------------------
# 7. Mira v2 augmentation
#
# No background/RIR datasets for this quick personalized pass.
# ------------------------------------------------------------

augmenter = Augmentation(
    augmentation_duration_s=3.2,

    augmentation_probabilities={
        "SevenBandParametricEQ": 0.15,
        "TanhDistortion": 0.10,
        "PitchShift": 0.15,
        "BandStopFilter": 0.10,
        "AddColorNoise": 0.30,

        "AddBackgroundNoise": 0.00,
        "RIR": 0.00,

        "Gain": 1.00,
        "GainTransition": 0.25,
    },

    impulse_paths=[],
    background_paths=[],

    background_min_snr_db=-5,
    background_max_snr_db=20,

    min_jitter_s=0.10,
    max_jitter_s=0.50,
)

print("Augmenter: OK")

# ------------------------------------------------------------
# 8. Split configurations
# ------------------------------------------------------------

STANDARD_SPLITS = {
    "training": {
        "split_name": "train",
        "repetition": 3,
        "slide_frames": 10,
    },

    "validation": {
        "split_name": "validation",
        "repetition": 1,
        "slide_frames": 10,
    },

    "testing": {
        "split_name": "test",
        "repetition": 1,
        "slide_frames": 1,
    },
}

# Much stronger representation of your actual voice.
REAL_VOICE_SPLITS = {
    "training": {
        "split_name": "train",
        "repetition": 25,
        "slide_frames": 10,
    },

    "validation": {
        "split_name": "validation",
        "repetition": 1,
        "slide_frames": 10,
    },

    "testing": {
        "split_name": "test",
        "repetition": 1,
        "slide_frames": 1,
    },
}

# ------------------------------------------------------------
# 9. Feature generation helper
# ------------------------------------------------------------

def build_feature_set(
    title,
    input_directory,
    output_directory,
    split_config,
):

    input_directory = Path(
        input_directory
    )

    output_directory = Path(
        output_directory
    )

    print()
    print("=" * 60)
    print(title)
    print("=" * 60)
    print(
        "Input WAVs:",
        wav_count(input_directory)
    )

    clips = Clips(
        input_directory=str(input_directory),
        file_pattern="*.wav",

        max_clip_duration_s=None,
        remove_silence=True,

        random_split_seed=42,
        split_count=0.1,
    )

    output_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    for split, cfg in split_config.items():

        split_dir = (
            output_directory /
            split
        )

        mmap_dir = (
            split_dir /
            "wakeword_mmap"
        )

        marker = (
            split_dir /
            "_MIRA_V2_COMPLETE.txt"
        )

        print()
        print("--------------------------------------")
        print(
            f"{title} / {split}"
        )
        print(
            "Repetition:",
            cfg["repetition"],
        )
        print(
            "Slide frames:",
            cfg["slide_frames"],
        )

        # Only skip something that we know finished.
        if (
            marker.exists()
            and mmap_dir.exists()
            and any(mmap_dir.iterdir())
        ):
            print("Already complete — skipping.")
            continue

        # Never trust a partial mmap from an interrupted run.
        if split_dir.exists():
            shutil.rmtree(
                split_dir
            )

        split_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        sg = SpectrogramGeneration(
            clips=clips,
            augmenter=augmenter,

            slide_frames=(
                cfg["slide_frames"]
            ),

            step_ms=10,
        )

        try:

            RaggedMmap.from_generator(
                out_dir=str(mmap_dir),

                batch_size=200,
                verbose=True,

                sample_generator=(
                    sg.spectrogram_generator(
                        split=(
                            cfg["split_name"]
                        ),

                        repeat=(
                            cfg["repetition"]
                        ),
                    )
                ),
            )

        except Exception:

            print()
            print(
                "Feature generation FAILED."
            )
            print(
                "Removing partial output:",
                mmap_dir,
            )

            if split_dir.exists():
                shutil.rmtree(
                    split_dir
                )

            raise

        marker.write_text(
            "Mira v2 feature extraction complete.\n",
            encoding="utf-8",
        )

        print()
        print(
            f"{split}: COMPLETE"
        )

# ------------------------------------------------------------
# 10. Synthetic Mira features
# ------------------------------------------------------------

build_feature_set(
    title="SYNTHETIC MIRA POSITIVES",

    input_directory=(
        synthetic_dir
    ),

    output_directory=(
        "/content/generated_augmented_features"
    ),

    split_config=(
        STANDARD_SPLITS
    ),
)

# ------------------------------------------------------------
# 11. Confusable negative features
# ------------------------------------------------------------

build_feature_set(
    title="CONFUSABLE NEGATIVES",

    input_directory=(
        confusable_dir
    ),

    output_directory=(
        "/content/confusable_features"
    ),

    split_config=(
        STANDARD_SPLITS
    ),
)

# ------------------------------------------------------------
# 12. Personalized real-voice features
# ------------------------------------------------------------

build_feature_set(
    title="REAL ALI MIRA POSITIVES",

    input_directory=(
        real_dir
    ),

    output_directory=(
        "/content/real_recording_features"
    ),

    split_config=(
        REAL_VOICE_SPLITS
    ),
)

# ------------------------------------------------------------
# 13. Final verification
# ------------------------------------------------------------

roots = {
    "Synthetic":
        Path(
            "/content/generated_augmented_features"
        ),

    "Confusable":
        Path(
            "/content/confusable_features"
        ),

    "Real voice":
        Path(
            "/content/real_recording_features"
        ),
}

print()
print("======================================")
print("MIRA V2 FEATURE EXTRACTION RESULTS")
print("======================================")

all_ready = True

for name, root in roots.items():

    ready = True

    for split in [
        "training",
        "validation",
        "testing",
    ]:

        marker = (
            root /
            split /
            "_MIRA_V2_COMPLETE.txt"
        )

        mmap_dir = (
            root /
            split /
            "wakeword_mmap"
        )

        if not (
            marker.exists()
            and mmap_dir.exists()
            and any(mmap_dir.iterdir())
        ):
            ready = False

    print(
        f"{name}:",
        "READY" if ready else "INCOMPLETE",
    )

    if not ready:
        all_ready = False

if not all_ready:
    raise RuntimeError(
        "One or more feature sets are incomplete."
    )

print()
print("======================================")
print("MIRA V2 FEATURES COMPLETE")
print("======================================")
print(
    "Synthetic positives: READY"
)
print(
    "Confusable negatives: READY"
)
print(
    "Personalized real voice: READY"
)
print()
print(
    "READY FOR PERSONALIZED TRAINING CONFIG"
)


MIRA V2 FEATURE INPUTS
Synthetic positives: 12000
Confusable negatives: 6400
Real Ali recordings: 52

Audiomentations: 0.43.1
Augmenter: OK

SYNTHETIC MIRA POSITIVES
Input WAVs: 12000

--------------------------------------
SYNTHETIC MIRA POSITIVES / training
Repetition: 3
Slide frames: 10


0it [00:00, ?it/s]


training: COMPLETE

--------------------------------------
SYNTHETIC MIRA POSITIVES / validation
Repetition: 1
Slide frames: 10


0it [00:00, ?it/s]


validation: COMPLETE

--------------------------------------
SYNTHETIC MIRA POSITIVES / testing
Repetition: 1
Slide frames: 1


...


training: COMPLETE

--------------------------------------
REAL ALI MIRA POSITIVES / validation
Repetition: 1
Slide frames: 10


0it [00:00, ?it/s]


validation: COMPLETE

--------------------------------------
REAL ALI MIRA POSITIVES / testing
Repetition: 1
Slide frames: 1


0it [00:00, ?it/s]


testing: COMPLETE

MIRA V2 FEATURE EXTRACTION RESULTS
Synthetic: READY
Confusable: READY
Real voice: READY

MIRA V2 FEATURES COMPLETE
Synthetic positives: READY
Confusable negatives: READY
Personalized real voice: READY

READY FOR PERSONALIZED TRAINING CONFIG


In [10]:
# === MIRA V2 — BACK UP FEATURE SETS TO GOOGLE DRIVE ===

import os
import shutil
import subprocess
from pathlib import Path

drive_root = Path(
    "/content/drive/MyDrive/wakeword_training_mira_v2"
)

backup_dir = drive_root / "feature_backups"
backup_dir.mkdir(parents=True, exist_ok=True)

feature_sets = {
    "generated_augmented_features":
        Path("/content/generated_augmented_features"),

    "confusable_features":
        Path("/content/confusable_features"),

    "real_recording_features":
        Path("/content/real_recording_features"),
}

print("======================================")
print("MIRA V2 FEATURE BACKUP")
print("======================================")

for name, source in feature_sets.items():

    if not source.exists():
        raise RuntimeError(
            f"Missing feature directory: {source}"
        )

    archive = backup_dir / f"{name}.tar.gz"

    print()
    print(f"Backing up: {name}")

    if archive.exists():
        archive.unlink()

    subprocess.run(
        [
            "tar",
            "-czf",
            str(archive),
            "-C",
            "/content",
            name,
        ],
        check=True,
    )

    size_mb = archive.stat().st_size / (1024 * 1024)

    print(
        f"Saved: {archive}"
    )
    print(
        f"Size: {size_mb:.1f} MB"
    )

print()
print("======================================")
print("FEATURE BACKUPS COMPLETE")
print("======================================")

for archive in sorted(backup_dir.glob("*.tar.gz")):
    print(
        archive.name,
        f"{archive.stat().st_size / (1024 * 1024):.1f} MB"
    )

MIRA V2 FEATURE BACKUP

Backing up: generated_augmented_features
Saved: /content/drive/MyDrive/wakeword_training_mira_v2/feature_backups/generated_augmented_features.tar.gz
Size: 925.1 MB

Backing up: confusable_features
Saved: /content/drive/MyDrive/wakeword_training_mira_v2/feature_backups/confusable_features.tar.gz
Size: 489.8 MB

Backing up: real_recording_features
Saved: /content/drive/MyDrive/wakeword_training_mira_v2/feature_backups/real_recording_features.tar.gz
Size: 31.0 MB

FEATURE BACKUPS COMPLETE
confusable_features.tar.gz 489.8 MB
generated_augmented_features.tar.gz 925.1 MB
real_recording_features.tar.gz 31.0 MB


In [1]:
# === MIRA V2 — A100 TRAINING RUNTIME RESTORE ===

import os
import sys
import shutil
import subprocess
from pathlib import Path

from google.colab import drive

os.chdir("/content")

# ------------------------------------------------------------
# 1. Mount Drive
# ------------------------------------------------------------

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

drive_root = Path(
    "/content/drive/MyDrive/wakeword_training_mira_v2"
)

feature_backup_dir = drive_root / "feature_backups"

# ------------------------------------------------------------
# 2. Confirm GPU
# ------------------------------------------------------------

print("======================================")
print("GPU CHECK")
print("======================================")

subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
    check=False,
)

# ------------------------------------------------------------
# 3. Install training dependencies
# ------------------------------------------------------------

print()
print("Installing microWakeWord dependencies...")

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "audiomentations==0.43.1",
        "audio_metadata",
        "datasets",
        "mmap_ninja",
        "numpy",
        "pymicro-features",
        "pyyaml",
        "tensorflow>=2.16",
        "webrtcvad-wheels",
        "ai-edge-litert",
        "numpy-minmax>=0.3.0,<1",
        "numpy-rms>=0.4.2,<1",
        "python-stretch>=0.3.1,<1",
        "soxr>=0.3.2,<1.0.0",
        "git+https://github.com/whatsnowplaying/audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f",
    ],
    check=True,
)

# ------------------------------------------------------------
# 4. Clone microWakeWord
# ------------------------------------------------------------

repo = Path("/content/microWakeWord")

if not repo.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "https://github.com/kahrendt/microWakeWord",
            str(repo),
        ],
        check=True,
    )

sys.path.insert(
    0,
    str(repo),
)

# ------------------------------------------------------------
# 5. Apply TensorFlow compatibility patch
# ------------------------------------------------------------

train_py = (
    repo /
    "microwakeword" /
    "train.py"
)

if train_py.exists():
    text = train_py.read_text(
        encoding="utf-8"
    )

    old = ".numpy()"

    if old in text:
        text = text.replace(
            ".numpy()",
            ".numpy() if hasattr(result, 'numpy') else result",
        )

        train_py.write_text(
            text,
            encoding="utf-8",
        )

# ------------------------------------------------------------
# 6. Restore feature archives
# ------------------------------------------------------------

archives = {
    "generated_augmented_features":
        feature_backup_dir /
        "generated_augmented_features.tar.gz",

    "confusable_features":
        feature_backup_dir /
        "confusable_features.tar.gz",

    "real_recording_features":
        feature_backup_dir /
        "real_recording_features.tar.gz",
}

print()
print("======================================")
print("RESTORING FEATURE SETS")
print("======================================")

for name, archive in archives.items():

    target = Path("/content") / name

    if target.exists():
        shutil.rmtree(target)

    if not archive.exists():
        raise RuntimeError(
            f"Missing backup archive: {archive}"
        )

    print(f"Restoring: {name}")

    subprocess.run(
        [
            "tar",
            "-xzf",
            str(archive),
            "-C",
            "/content",
        ],
        check=True,
    )

    print(f"Restored: {target}")

# ------------------------------------------------------------
# 7. Final verification
# ------------------------------------------------------------

print()
print("======================================")
print("MIRA V2 A100 RESTORE COMPLETE")
print("======================================")

for name in archives:
    path = Path("/content") / name

    print(
        f"{name}:",
        "READY" if path.exists() else "MISSING"
    )

print()
print("READY TO CREATE TRAINING CONFIG")

Mounted at /content/drive
GPU CHECK

Installing microWakeWord dependencies...

RESTORING FEATURE SETS
Restoring: generated_augmented_features
Restored: /content/generated_augmented_features
Restoring: confusable_features
Restored: /content/confusable_features
Restoring: real_recording_features
Restored: /content/real_recording_features

MIRA V2 A100 RESTORE COMPLETE
generated_augmented_features: READY
confusable_features: READY
real_recording_features: READY

READY TO CREATE TRAINING CONFIG


In [2]:
# === MIRA V2 — CREATE PERSONALIZED TRAINING CONFIG ===

from pathlib import Path
import yaml

config = {
    "window_step_ms": 10,

    "train_dir": "trained_models/mira_v2",

    "features": [
        {
            "features_dir": "generated_augmented_features",
            "sampling_weight": 8.0,
            "penalty_weight": 2.0,
            "truth": True,
            "truncation_strategy": "truncate_start",
            "type": "mmap",
        },

        {
            "features_dir": "real_recording_features",
            "sampling_weight": 24.0,
            "penalty_weight": 2.0,
            "truth": True,
            "truncation_strategy": "truncate_start",
            "type": "mmap",
        },

        {
            "features_dir": "confusable_features",
            "sampling_weight": 10.0,
            "penalty_weight": 5.0,
            "truth": False,
            "truncation_strategy": "random",
            "type": "mmap",
        },
    ],

    # 25,000 total steps
    "training_steps": [
        15000,
        10000,
    ],

    "positive_class_weight": [
        2,
        2,
    ],

    "negative_class_weight": [
        40,
        50,
    ],

    "learning_rates": [
        0.001,
        0.0001,
    ],

    "batch_size": 256,

    "time_mask_max_size": [
        5,
        5,
    ],

    "time_mask_count": [
        1,
        1,
    ],

    "freq_mask_max_size": [
        3,
        3,
    ],

    "freq_mask_count": [
        1,
        1,
    ],

    "eval_step_interval": 500,

    "clip_duration_ms": 1500,

    # No ambient validation dataset in this quick personalized v2 pass,
    # so use validation loss + recall instead.
    "target_minimization": 0.40,
    "minimization_metric": "loss",
    "maximization_metric": "recall",
}

config_path = Path(
    "/content/mira_v2_training_parameters.yaml"
)

with open(config_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(
        config,
        f,
        sort_keys=False,
    )

# Save a persistent copy to Drive immediately
drive_copy = Path(
    "/content/drive/MyDrive/"
    "wakeword_training_mira_v2/"
    "mira_v2_training_parameters.yaml"
)

drive_copy.write_text(
    config_path.read_text(encoding="utf-8"),
    encoding="utf-8",
)

print("======================================")
print("MIRA V2 TRAINING CONFIG CREATED")
print("======================================")
print()
print(config_path.read_text())
print("Saved locally:", config_path)
print("Saved to Drive:", drive_copy)

MIRA V2 TRAINING CONFIG CREATED

window_step_ms: 10
train_dir: trained_models/mira_v2
features:
- features_dir: generated_augmented_features
  sampling_weight: 8.0
  penalty_weight: 2.0
  truth: true
  truncation_strategy: truncate_start
  type: mmap
- features_dir: real_recording_features
  sampling_weight: 24.0
  penalty_weight: 2.0
  truth: true
  truncation_strategy: truncate_start
  type: mmap
- features_dir: confusable_features
  sampling_weight: 10.0
  penalty_weight: 5.0
  truth: false
  truncation_strategy: random
  type: mmap
training_steps:
- 15000
- 10000
positive_class_weight:
- 2
- 2
negative_class_weight:
- 40
- 50
learning_rates:
- 0.001
- 0.0001
batch_size: 256
time_mask_max_size:
- 5
- 5
time_mask_count:
- 1
- 1
freq_mask_max_size:
- 3
- 3
freq_mask_count:
- 1
- 1
eval_step_interval: 500
clip_duration_ms: 1500
target_minimization: 0.4
minimization_metric: loss
maximization_metric: recall

Saved locally: /content/mira_v2_training_parameters.yaml
Saved to Drive: /conten

In [3]:
# === MIRA V2 — START PERSONALIZED TRAINING ===

import os
import sys
import subprocess
from pathlib import Path

os.chdir("/content")

repo = Path("/content/microWakeWord")
config_path = Path("/content/mira_v2_training_parameters.yaml")

if not repo.exists():
    raise RuntimeError("microWakeWord repo is missing.")

if not config_path.exists():
    raise RuntimeError("Mira v2 training config is missing.")

print("======================================")
print("MIRA V2 PERSONALIZED TRAINING")
print("======================================")
print("Config:", config_path)
print("GPU:")
subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    check=False,
)

print()
print("Training architecture:")
print("  inception")
print("  cnn1 filters: 32")
print("  cnn2 filters: 24/32, 24/64, 24/96")
print("  dropout: 0.8")
print()
print("Training steps: 25,000 total")
print("Stage 1: 15,000 @ 0.001")
print("Stage 2: 10,000 @ 0.0001")
print()

cmd = [
    sys.executable,
    "-m",
    "microwakeword.train",

    "--training_config",
    str(config_path),

    "inception",

    "--cnn1_filters",
    "32",

    "--cnn1_kernel_sizes",
    "5",

    "--cnn1_subspectral_groups",
    "4",

    "--cnn2_filters1",
    "24,24,24",

    "--cnn2_filters2",
    "32,64,96",

    "--cnn2_kernel_sizes",
    "3,5,5",

    "--cnn2_subspectral_groups",
    "1,1,1",

    "--cnn2_dilation",
    "1,1,1",

    "--dropout",
    "0.8",
]

print("Launching training...")
print()

result = subprocess.run(
    cmd,
    cwd=str(repo),
)

print()
print("======================================")
print("TRAINING PROCESS FINISHED")
print("======================================")
print("Exit code:", result.returncode)

if result.returncode != 0:
    raise RuntimeError(
        f"Mira v2 training failed with exit code {result.returncode}"
    )

print()
print("MIRA V2 TRAINING COMPLETED")

MIRA V2 PERSONALIZED TRAINING
Config: /content/mira_v2_training_parameters.yaml
GPU:

Training architecture:
  inception
  cnn1 filters: 32
  cnn2 filters: 24/32, 24/64, 24/96
  dropout: 0.8

Training steps: 25,000 total
Stage 1: 15,000 @ 0.001
Stage 2: 10,000 @ 0.0001

Launching training...


TRAINING PROCESS FINISHED
Exit code: 0

MIRA V2 TRAINING COMPLETED


In [8]:
# === MIRA V2 — ACTUAL PERSONALIZED TRAINING + AUTO BACKUP ===

import os
import sys
import shutil
import subprocess
from pathlib import Path

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

repo = Path("/content/microWakeWord")
config_path = Path("/content/mira_v2_training_parameters.yaml")

train_dir = Path("/content/trained_models/mira_v2")

drive_train_dir = Path(
    "/content/drive/MyDrive/"
    "wakeword_training_mira_v2/"
    "trained_models/"
    "mira_v2"
)

# ------------------------------------------------------------
# PREFLIGHT
# ------------------------------------------------------------

print("======================================")
print("MIRA V2 REAL TRAINING PREFLIGHT")
print("======================================")

required = [
    repo,
    config_path,
    Path("/content/generated_augmented_features"),
    Path("/content/confusable_features"),
    Path("/content/real_recording_features"),
]

for p in required:
    status = "READY" if p.exists() else "MISSING"
    print(f"{p}: {status}")

    if not p.exists():
        raise RuntimeError(
            f"Required path is missing: {p}"
        )

# We should NOT already have a trained model from the false launch.
if train_dir.exists():
    raise RuntimeError(
        f"{train_dir} already exists. "
        "Stop here so we can inspect it before overwriting anything."
    )

print()
print("GPU:")
subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=name,memory.total",
        "--format=csv,noheader",
    ],
    check=False,
)

# ------------------------------------------------------------
# MAKE MICROWAKEWORD IMPORTABLE WHILE WORKING FROM /content
# ------------------------------------------------------------

env = os.environ.copy()

existing_pythonpath = env.get("PYTHONPATH", "")

env["PYTHONPATH"] = (
    str(repo)
    if not existing_pythonpath
    else str(repo) + os.pathsep + existing_pythonpath
)

# ------------------------------------------------------------
# REAL TRAINING COMMAND
# ------------------------------------------------------------

cmd = [
    sys.executable,
    "-m",
    "microwakeword.model_train_eval",

    "--training_config",
    str(config_path),

    "--train",
    "1",

    # Don't export yet. Train first and preserve weights.
    "--test_tf_nonstreaming",
    "0",

    "--test_tflite_nonstreaming",
    "0",

    "--test_tflite_nonstreaming_quantized",
    "0",

    "--test_tflite_streaming",
    "0",

    "--test_tflite_streaming_quantized",
    "0",

    "inception",

    "--cnn1_filters",
    "32",

    "--cnn1_kernel_sizes",
    "5",

    "--cnn1_subspectral_groups",
    "4",

    "--cnn2_filters1",
    "24,24,24",

    "--cnn2_filters2",
    "32,64,96",

    "--cnn2_kernel_sizes",
    "3,5,5",

    "--cnn2_subspectral_groups",
    "1,1,1",

    "--cnn2_dilation",
    "1,1,1",

    "--dropout",
    "0.8",
]

print()
print("======================================")
print("STARTING ACTUAL MIRA V2 TRAINING")
print("======================================")
print("25,000 steps total")
print("Stage 1: 15,000 @ 0.001")
print("Stage 2: 10,000 @ 0.0001")
print()

result = subprocess.run(
    cmd,
    cwd="/content",
    env=env,
)

print()
print("Training exit code:", result.returncode)

if result.returncode != 0:
    raise RuntimeError(
        f"Training failed with exit code {result.returncode}"
    )

# ------------------------------------------------------------
# VERIFY TRAINING REALLY PRODUCED WEIGHTS
# ------------------------------------------------------------

best_weights = (
    train_dir /
    "best_weights.weights.h5"
)

last_weights = (
    train_dir /
    "last_weights.weights.h5"
)

print()
print("======================================")
print("VERIFYING TRAINING ARTIFACTS")
print("======================================")

print(
    "Training directory:",
    "FOUND" if train_dir.exists() else "MISSING"
)

print(
    "Best weights:",
    "FOUND" if best_weights.exists() else "MISSING"
)

print(
    "Last weights:",
    "FOUND" if last_weights.exists() else "MISSING"
)

if not train_dir.exists():
    raise RuntimeError(
        "Training returned success but no training directory was created."
    )

if not last_weights.exists():
    raise RuntimeError(
        "Training returned success but last_weights.weights.h5 is missing."
    )

# ------------------------------------------------------------
# IMMEDIATELY BACK UP EVERYTHING TO DRIVE
# ------------------------------------------------------------

print()
print("======================================")
print("BACKING UP TO GOOGLE DRIVE")
print("======================================")

if drive_train_dir.exists():
    shutil.rmtree(drive_train_dir)

drive_train_dir.parent.mkdir(
    parents=True,
    exist_ok=True,
)

shutil.copytree(
    train_dir,
    drive_train_dir,
)

# Preserve our original personalized YAML too.
shutil.copy2(
    config_path,
    drive_train_dir /
    "mira_v2_training_parameters.yaml",
)

print("Drive backup:", drive_train_dir)

# ------------------------------------------------------------
# FINAL SUMMARY
# ------------------------------------------------------------

print()
print("======================================")
print("MIRA V2 ACTUAL TRAINING COMPLETE")
print("======================================")

for p in sorted(train_dir.rglob("*")):
    if p.is_file():
        print(
            p.relative_to(train_dir),
            f"{p.stat().st_size / 1024:.1f} KB"
        )

print()
print("LOCAL TRAINING: VERIFIED")
print("DRIVE BACKUP:   VERIFIED")
print()
print("READY FOR TFLITE EXPORT")

MIRA V2 REAL TRAINING PREFLIGHT
/content/microWakeWord: READY
/content/mira_v2_training_parameters.yaml: READY
/content/generated_augmented_features: READY
/content/confusable_features: READY
/content/real_recording_features: READY

GPU:

STARTING ACTUAL MIRA V2 TRAINING
25,000 steps total
Stage 1: 15,000 @ 0.001
Stage 2: 10,000 @ 0.0001


Training exit code: 0

VERIFYING TRAINING ARTIFACTS
Training directory: FOUND
Best weights: FOUND
Last weights: FOUND

BACKING UP TO GOOGLE DRIVE
Drive backup: /content/drive/MyDrive/wakeword_training_mira_v2/trained_models/mira_v2

MIRA V2 ACTUAL TRAINING COMPLETE
best_weights.weights.h5 1105.2 KB
last_weights.weights.h5 1105.2 KB
logs/train/events.out.tfevents.1788585421.174be71d1563.12481.0.v2 16.0 KB
logs/validation/events.out.tfevents.1788585421.174be71d1563.12481.1.v2 23.7 KB
model_summary.txt 30.5 KB
restore/checkpoint 0.1 KB
restore/ckpt-1.data-00000-of-00001 834.2 KB
restore/ckpt-1.index 14.4 KB
restore/ckpt-2.data-00000-of-00001 834.2 KB
re

In [9]:
# === MIRA V2 — EXPORT + VERIFY + SAVE FINAL TFLITE ===

import os
import sys
import json
import shutil
import hashlib
import subprocess
from pathlib import Path

repo = Path("/content/microWakeWord")
config = Path("/content/mira_v2_training_parameters.yaml")

train_dir = Path("/content/trained_models/mira_v2")

best_weights = train_dir / "best_weights.weights.h5"

drive_root = Path(
    "/content/drive/MyDrive/wakeword_training_mira_v2"
)

# ------------------------------------------------------------
# PREFLIGHT
# ------------------------------------------------------------

print("======================================")
print("MIRA V2 EXPORT PREFLIGHT")
print("======================================")

for p in [repo, config, train_dir, best_weights]:
    print(p, "READY" if p.exists() else "MISSING")

    if not p.exists():
        raise RuntimeError(f"Missing required path: {p}")

# Make repo importable while staying in /content so relative
# feature paths in our training config remain correct.
env = os.environ.copy()

existing = env.get("PYTHONPATH", "")

env["PYTHONPATH"] = (
    str(repo)
    if not existing
    else str(repo) + os.pathsep + existing
)

# ------------------------------------------------------------
# EXPORT QUANTIZED STREAMING MODEL
# ------------------------------------------------------------

cmd = [
    sys.executable,
    "-m",
    "microwakeword.model_train_eval",

    "--training_config",
    str(config),

    "--train",
    "0",

    "--use_weights",
    "best_weights",

    "--test_tf_nonstreaming",
    "0",

    "--test_tflite_nonstreaming",
    "0",

    "--test_tflite_nonstreaming_quantized",
    "0",

    "--test_tflite_streaming",
    "0",

    "--test_tflite_streaming_quantized",
    "1",

    "inception",

    "--cnn1_filters",
    "32",

    "--cnn1_kernel_sizes",
    "5",

    "--cnn1_subspectral_groups",
    "4",

    "--cnn2_filters1",
    "24,24,24",

    "--cnn2_filters2",
    "32,64,96",

    "--cnn2_kernel_sizes",
    "3,5,5",

    "--cnn2_subspectral_groups",
    "1,1,1",

    "--cnn2_dilation",
    "1,1,1",

    "--dropout",
    "0.8",
]

print()
print("======================================")
print("EXPORTING MIRA V2")
print("======================================")
print("Using: best_weights.weights.h5")
print()

result = subprocess.run(
    cmd,
    cwd="/content",
    env=env,
)

print()
print("Export exit code:", result.returncode)

if result.returncode != 0:
    raise RuntimeError(
        f"Mira v2 export failed with exit code {result.returncode}"
    )

# ------------------------------------------------------------
# LOCATE GENERATED TFLITE
# ------------------------------------------------------------

generated = (
    train_dir
    / "tflite_stream_state_internal_quant"
    / "stream_state_internal_quant.tflite"
)

if not generated.exists():
    raise RuntimeError(
        f"Export completed but TFLite was not found: {generated}"
    )

final_model = Path("/content/mira_v2.tflite")

shutil.copy2(
    generated,
    final_model,
)

print()
print("Generated model:")
print(generated)

# ------------------------------------------------------------
# VERIFY WITH LITERT / TFLITE INTERPRETER
# ------------------------------------------------------------

try:
    from ai_edge_litert.interpreter import Interpreter
except ImportError:
    from tensorflow.lite.python.interpreter import Interpreter

interpreter = Interpreter(
    model_path=str(final_model)
)

interpreter.allocate_tensors()

inputs = interpreter.get_input_details()
outputs = interpreter.get_output_details()

print()
print("======================================")
print("TFLITE VERIFICATION")
print("======================================")

print("Input shape :", inputs[0]["shape"])
print("Input dtype :", inputs[0]["dtype"])

print("Output shape:", outputs[0]["shape"])
print("Output dtype:", outputs[0]["dtype"])

print(
    "Output quantization:",
    outputs[0]["quantization"]
)

input_shape = list(inputs[0]["shape"])
output_shape = list(outputs[0]["shape"])

input_dtype = str(inputs[0]["dtype"])
output_dtype = str(outputs[0]["dtype"])

if input_shape != [1, 1, 40]:
    raise RuntimeError(
        f"Unexpected input shape: {input_shape}"
    )

if output_shape != [1, 1]:
    raise RuntimeError(
        f"Unexpected output shape: {output_shape}"
    )

if "int8" not in input_dtype or "uint8" in input_dtype:
    raise RuntimeError(
        f"Unexpected input dtype: {input_dtype}"
    )

if "uint8" not in output_dtype:
    raise RuntimeError(
        f"Unexpected output dtype: {output_dtype}"
    )

# ------------------------------------------------------------
# SHA256
# ------------------------------------------------------------

sha256 = hashlib.sha256(
    final_model.read_bytes()
).hexdigest().upper()

print()
print("Size   :", final_model.stat().st_size, "bytes")
print("SHA256 :", sha256)

# ------------------------------------------------------------
# CREATE MIRA MANIFEST
# ------------------------------------------------------------

manifest = {
    "type": "micro",
    "wake_word": "Mira",
    "author": "Ali / Rin",
    "website": "",
    "model": "mira_v2.tflite",
    "trained_languages": ["en"],
    "version": 2,
    "micro": {
        "probability_cutoff": 0.85,
        "sliding_window_size": 5,
        "feature_step_size": 10,
    },
}

manifest_path = Path("/content/mira_v2.json")

manifest_path.write_text(
    json.dumps(
        manifest,
        indent=2,
    ),
    encoding="utf-8",
)

# ------------------------------------------------------------
# SAVE FINAL FILES TO DRIVE
# ------------------------------------------------------------

drive_model = drive_root / "mira_v2.tflite"
drive_manifest = drive_root / "mira_v2.json"

shutil.copy2(
    final_model,
    drive_model,
)

shutil.copy2(
    manifest_path,
    drive_manifest,
)

print()
print("======================================")
print("MIRA V2 EXPORT COMPLETE")
print("======================================")

print("MODEL:")
print(final_model)

print()
print("MANIFEST:")
print(manifest_path)

print()
print("DRIVE MODEL:")
print(drive_model)

print()
print("DRIVE MANIFEST:")
print(drive_manifest)

print()
print("======================================")
print("ANDROID COMPATIBILITY: VERIFIED")
print("======================================")
print("[1,1,40] INT8 -> [1,1] UINT8")
print()
print("READY TO INSTALL ON RIN MOBILE")

MIRA V2 EXPORT PREFLIGHT
/content/microWakeWord READY
/content/mira_v2_training_parameters.yaml READY
/content/trained_models/mira_v2 READY
/content/trained_models/mira_v2/best_weights.weights.h5 READY

EXPORTING MIRA V2
Using: best_weights.weights.h5


Export exit code: 0

Generated model:
/content/trained_models/mira_v2/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite

TFLITE VERIFICATION
Input shape : [ 1  1 40]
Input dtype : <class 'numpy.int8'>
Output shape: [1 1]
Output dtype: <class 'numpy.uint8'>
Output quantization: (0.00390625, 0)

Size   : 127112 bytes
SHA256 : 1B9F66E8772007BE8CCD285670CD4637250050154D55C363FAF61683E5ED370F

MIRA V2 EXPORT COMPLETE
MODEL:
/content/mira_v2.tflite

MANIFEST:
/content/mira_v2.json

DRIVE MODEL:
/content/drive/MyDrive/wakeword_training_mira_v2/mira_v2.tflite

DRIVE MANIFEST:
/content/drive/MyDrive/wakeword_training_mira_v2/mira_v2.json

ANDROID COMPATIBILITY: VERIFIED
[1,1,40] INT8 -> [1,1] UINT8

READY TO INSTALL ON RIN MO

In [7]:
# === MIRA V2 — LOCATE + BACK UP TRAINING ARTIFACTS ===

import shutil
from pathlib import Path

# The training command ran with cwd=/content/microWakeWord
expected_dir = Path(
    "/content/microWakeWord/trained_models/mira_v2"
)

# Fall back to searching if anything differs
candidates = [
    expected_dir,
    Path("/content/trained_models/mira_v2"),
]

train_dir = None

for candidate in candidates:
    if candidate.exists():
        train_dir = candidate
        break

if train_dir is None:
    print("Searching for mira_v2 training directory...")

    matches = [
        p for p in Path("/content").rglob("mira_v2")
        if p.is_dir()
    ]

    print()
    print("Found:")
    for p in matches:
        print(p)

    # Prefer a mira_v2 directory underneath trained_models
    matches = [
        p for p in matches
        if p.parent.name == "trained_models"
    ]

    if not matches:
        raise RuntimeError(
            "Could not locate the Mira v2 trained model directory."
        )

    train_dir = matches[0]

print()
print("======================================")
print("MIRA V2 TRAINING DIRECTORY")
print("======================================")
print(train_dir)

drive_dir = Path(
    "/content/drive/MyDrive/"
    "wakeword_training_mira_v2/"
    "trained_models/"
    "mira_v2"
)

drive_dir.mkdir(
    parents=True,
    exist_ok=True,
)

print()
print("======================================")
print("BACKING UP TRAINING ARTIFACTS")
print("======================================")

for item in train_dir.iterdir():

    destination = drive_dir / item.name

    if item.is_dir():

        if destination.exists():
            shutil.rmtree(destination)

        shutil.copytree(
            item,
            destination,
        )

    else:

        shutil.copy2(
            item,
            destination,
        )

    print("Saved:", item.name)

# Preserve exact YAML too
config_src = Path(
    "/content/mira_v2_training_parameters.yaml"
)

if config_src.exists():
    shutil.copy2(
        config_src,
        drive_dir / "mira_v2_training_parameters.yaml",
    )

print()
print("======================================")
print("BACKUP COMPLETE")
print("======================================")
print("Source:", train_dir)
print("Drive :", drive_dir)

print()
print("ARTIFACT LIST:")
print()

for p in sorted(drive_dir.rglob("*")):

    if p.is_file():

        print(
            p.relative_to(drive_dir),
            f"{p.stat().st_size / 1024:.1f} KB"
        )

Searching for mira_v2 training directory...

Found:


RuntimeError: Could not locate the Mira v2 trained model directory.

In [6]:
# === MIRA V2 — BUILD TRAINING FEATURES ===

import os
import sys
import re
import shutil
import subprocess
import importlib
import traceback
from pathlib import Path

os.chdir("/content")

# ============================================================
# 1. Make sure microWakeWord training environment exists
# ============================================================

DEPS = [
    "audiomentations",
    "audio_metadata",
    "datasets",
    "mmap_ninja",
    "numpy",
    "pymicro-features",
    "pyyaml",
    "tensorflow>=2.16",
    "webrtcvad-wheels",
    "ai-edge-litert",
    (
        "git+https://github.com/whatsnowplaying/"
        "audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f"
    ),
]

print("Checking/installing microWakeWord dependencies...")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q"] + DEPS,
    check=True,
)

MWW_DIR = Path("/content/microWakeWord")

if not MWW_DIR.exists():
    print("Cloning microWakeWord...")

    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "https://github.com/kahrendt/microWakeWord",
            str(MWW_DIR),
        ],
        check=True,
    )

if str(MWW_DIR) not in sys.path:
    sys.path.insert(0, str(MWW_DIR))

importlib.invalidate_caches()

# ------------------------------------------------------------
# TensorFlow compatibility patch
# ------------------------------------------------------------

train_py = (
    MWW_DIR /
    "microwakeword/train.py"
)

src = train_py.read_text()

patched = re.sub(
    r'(\b[a-zA-Z_]+\["[a-z]+"\])\.numpy\(\)',
    r'(\1.numpy() if hasattr(\1, "numpy") else \1)',
    src,
)

if patched != src:
    train_py.write_text(patched)
    print("Applied TensorFlow compatibility patch.")

# ------------------------------------------------------------
# Imports
# ------------------------------------------------------------

from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration
from mmap_ninja.ragged import RaggedMmap

print("microWakeWord imports: OK")

# ============================================================
# 2. Locate/recover all three datasets
# ============================================================

drive_root = Path(
    "/content/drive/MyDrive/wakeword_training_mira_v2"
)

backup_root = (
    drive_root /
    "generated_backups"
)

synthetic_dir = Path(
    "/content/generated_samples_v2"
)

confusable_dir = Path(
    "/content/confusable_negatives_v2_flat"
)

real_drive_dir = (
    drive_root /
    "real_voice"
)

real_dir = Path(
    "/content/real_recordings"
)

# ------------------------------------------------------------
# Recover synthetic data from Drive if runtime was reset
# ------------------------------------------------------------

synthetic_count = (
    len(list(synthetic_dir.glob("*.wav")))
    if synthetic_dir.exists()
    else 0
)

if synthetic_count != 12000:
    print()
    print("Recovering 12,000 synthetic Mira samples from Drive...")

    if synthetic_dir.exists():
        shutil.rmtree(synthetic_dir)

    archive = (
        backup_root /
        "synthetic_mira_12000.tar.gz"
    )

    if not archive.exists():
        raise RuntimeError(
            f"Missing synthetic backup: {archive}"
        )

    subprocess.run(
        [
            "tar",
            "-xzf",
            str(archive),
            "-C",
            "/content",
        ],
        check=True,
    )

# ------------------------------------------------------------
# Recover confusables if needed
# ------------------------------------------------------------

confusable_count = (
    len(list(confusable_dir.glob("*.wav")))
    if confusable_dir.exists()
    else 0
)

if confusable_count != 6400:
    print()
    print("Recovering 6,400 confusable samples from Drive...")

    if confusable_dir.exists():
        shutil.rmtree(confusable_dir)

    archive = (
        backup_root /
        "confusable_negatives_6400.tar.gz"
    )

    if not archive.exists():
        raise RuntimeError(
            f"Missing confusable backup: {archive}"
        )

    subprocess.run(
        [
            "tar",
            "-xzf",
            str(archive),
            "-C",
            "/content",
        ],
        check=True,
    )

# ------------------------------------------------------------
# Copy real recordings from Drive into fast local storage
# ------------------------------------------------------------

if real_dir.exists():
    shutil.rmtree(real_dir)

shutil.copytree(
    real_drive_dir,
    real_dir,
)

# ------------------------------------------------------------
# Final input verification
# ------------------------------------------------------------

synthetic_wavs = sorted(
    synthetic_dir.glob("*.wav")
)

confusable_wavs = sorted(
    confusable_dir.glob("*.wav")
)

real_wavs = sorted(
    real_dir.glob("*.wav")
)

print()
print("======================================")
print("MIRA V2 FEATURE INPUTS")
print("======================================")
print("Synthetic positives:", len(synthetic_wavs))
print("Confusable negatives:", len(confusable_wavs))
print("Real Ali recordings:", len(real_wavs))

assert len(synthetic_wavs) == 12000
assert len(confusable_wavs) == 6400
assert len(real_wavs) == 52

# ============================================================
# 3. Minimal augmentation
#
# No external ambient dataset or RIR for this quick v2.
# ============================================================

augmenter = Augmentation(
    augmentation_duration_s=3.2,

    augmentation_probabilities={
        "SevenBandParametricEQ": 0.15,
        "TanhDistortion": 0.10,
        "PitchShift": 0.15,
        "BandStopFilter": 0.10,
        "AddColorNoise": 0.30,

        # Deliberately disabled for this v2 pass
        "AddBackgroundNoise": 0.0,
        "RIR": 0.0,

        "Gain": 1.00,
        "GainTransition": 0.25,
    },

    impulse_paths=[],
    background_paths=[],

    background_min_snr_db=-5,
    background_max_snr_db=20,

    min_jitter_s=0.10,
    max_jitter_s=0.50,
)

# ============================================================
# 4. Feature extraction helper
# ============================================================

NORMAL_SPLITS = {
    "training": {
        "split_name": "train",
        "repetition": 3,
        "slide_frames": 10,
    },
    "validation": {
        "split_name": "validation",
        "repetition": 1,
        "slide_frames": 10,
    },
    "testing": {
        "split_name": "test",
        "repetition": 1,
        "slide_frames": 1,
    },
}

# Your voice receives many more augmented repetitions.
REAL_SPLITS = {
    "training": {
        "split_name": "train",
        "repetition": 25,
        "slide_frames": 10,
    },
    "validation": {
        "split_name": "validation",
        "repetition": 1,
        "slide_frames": 10,
    },
    "testing": {
        "split_name": "test",
        "repetition": 1,
        "slide_frames": 1,
    },
}


def build_features(
    label,
    input_directory,
    output_root,
    split_config,
):
    print()
    print("=" * 60)
    print(label)
    print("=" * 60)

    clips = Clips(
        input_directory=str(input_directory),
        file_pattern="*.wav",
        max_clip_duration_s=None,
        remove_silence=True,
        random_split_seed=42,
        split_count=0.1,
    )

    output_root = Path(output_root)
    output_root.mkdir(
        parents=True,
        exist_ok=True,
    )

    for split, cfg in split_config.items():

        out = (
            output_root /
            split
        )

        mmap_dir = (
            out /
            "wakeword_mmap"
        )

        complete_marker = (
            out /
            "_MIRA_FEATURES_COMPLETE.txt"
        )

        # Only trust an earlier run if it completed fully.
        if (
            complete_marker.exists()
            and mmap_dir.exists()
        ):
            print(
                f"{split}: already complete, skipping."
            )
            continue

        # Remove incomplete previous attempts
        if out.exists():
            shutil.rmtree(out)

        out.mkdir(
            parents=True,
            exist_ok=True,
        )

        print()
        print(
            f"{split}: "
            f"repeat={cfg['repetition']}, "
            f"slide={cfg['slide_frames']}"
        )

        try:
            sg = SpectrogramGeneration(
                clips=clips,
                augmenter=augmenter,
                slide_frames=cfg["slide_frames"],
                step_ms=10,
            )

            RaggedMmap.from_generator(
                out_dir=str(mmap_dir),
                batch_size=200,
                verbose=True,
                sample_generator=(
                    sg.spectrogram_generator(
                        split=cfg["split_name"],
                        repeat=cfg["repetition"],
                    )
                ),
            )

            complete_marker.write_text(
                "complete\n"
            )

            print(
                f"{split}: COMPLETE"
            )

        except Exception:
            traceback.print_exc()

            if out.exists():
                shutil.rmtree(out)

            raise

# ============================================================
# 5. Synthetic Mira positive features
# ============================================================

build_features(
    label="SYNTHETIC MIRA POSITIVES",
    input_directory=synthetic_dir,
    output_root="/content/generated_augmented_features",
    split_config=NORMAL_SPLITS,
)

# ============================================================
# 6. Confusable negative features
# ============================================================

build_features(
    label="CONFUSABLE NEGATIVES",
    input_directory=confusable_dir,
    output_root="/content/confusable_features",
    split_config=NORMAL_SPLITS,
)

# ============================================================
# 7. Your real voice features
#
# 25x training augmentation is intentional.
# ============================================================

build_features(
    label="REAL ALI MIRA RECORDINGS",
    input_directory=real_dir,
    output_root="/content/real_recording_features",
    split_config=REAL_SPLITS,
)

# ============================================================
# 8. Final check
# ============================================================

feature_roots = [
    Path("/content/generated_augmented_features"),
    Path("/content/confusable_features"),
    Path("/content/real_recording_features"),
]

print()
print("======================================")
print("MIRA V2 FEATURES COMPLETE")
print("======================================")

for root in feature_roots:

    complete = all(
        (
            root /
            split /
            "_MIRA_FEATURES_COMPLETE.txt"
        ).exists()
        for split in [
            "training",
            "validation",
            "testing",
        ]
    )

    print(
        root.name,
        "READY" if complete else "INCOMPLETE"
    )

    if not complete:
        raise RuntimeError(
            f"Incomplete feature set: {root}"
        )

print()
print("======================================")
print("READY FOR MIRA V2 TRAINING CONFIG")
print("======================================")

Checking/installing microWakeWord dependencies...
Cloning microWakeWord...
microWakeWord imports: OK

MIRA V2 FEATURE INPUTS
Synthetic positives: 12000
Confusable negatives: 6400
Real Ali recordings: 52


AttributeError: module 'audiomentations' has no attribute 'AddColorNoise'

In [1]:
# === MIRA V2 — INSTALL FULL OFFICIAL LIBRITTS-R CONFIG + TEST ===

import os
import sys
import json
import shutil
import subprocess
import requests
from pathlib import Path

os.chdir("/content")

model_path = Path(
    "/content/models/en_US-libritts_r-medium.pt"
)

config_path = Path(
    "/content/models/en_US-libritts_r-medium.pt.json"
)

test_dir = Path(
    "/content/mira_v2_test"
)

if not model_path.exists():
    raise RuntimeError(
        f"Missing model: {model_path}"
    )

# ------------------------------------------------------------
# Download the FULL official LibriTTS-R Piper config
# ------------------------------------------------------------

config_url = (
    "https://huggingface.co/rhasspy/piper-voices/"
    "raw/main/en/en_US/libritts_r/medium/"
    "en_US-libritts_r-medium.onnx.json"
)

print("Downloading full official LibriTTS-R config...")

r = requests.get(
    config_url,
    timeout=60
)

r.raise_for_status()

config_path.write_bytes(r.content)

print(
    "Saved:",
    config_path,
    f"({config_path.stat().st_size / 1024:.1f} KB)"
)

# ------------------------------------------------------------
# Validate the important fields
# ------------------------------------------------------------

with config_path.open(
    "r",
    encoding="utf-8"
) as f:
    config = json.load(f)

required = [
    "audio",
    "espeak",
    "phoneme_id_map",
    "num_symbols",
    "num_speakers",
]

missing = [
    key
    for key in required
    if key not in config
]

if missing:
    raise RuntimeError(
        f"Config missing required fields: {missing}"
    )

print()
print("=== CONFIG CHECK ===")
print(
    "Sample rate:",
    config["audio"]["sample_rate"]
)
print(
    "eSpeak voice:",
    config["espeak"]["voice"]
)
print(
    "Num symbols:",
    config["num_symbols"]
)
print(
    "Num speakers:",
    config["num_speakers"]
)
print(
    "Phoneme map entries:",
    len(config["phoneme_id_map"])
)

assert config["audio"]["sample_rate"] == 22050
assert config["num_speakers"] == 904
assert len(config["phoneme_id_map"]) > 100

# ------------------------------------------------------------
# Environment
# ------------------------------------------------------------

env = os.environ.copy()

env["PYTHONPATH"] = ":".join([
    "/content/piper/src/python",
    "/content/piper-sample-generator",
    env.get("PYTHONPATH", ""),
])

# ------------------------------------------------------------
# Clean previous failed test
# ------------------------------------------------------------

if test_dir.exists():
    shutil.rmtree(test_dir)

test_dir.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# Run 10-sample test
# ------------------------------------------------------------

cmd = [
    sys.executable,
    "-m",
    "piper_sample_generator",
    "Mira",
    "--model",
    str(model_path),
    "--max-samples",
    "10",
    "--batch-size",
    "8",
    "--max-speakers",
    "50",
    "--length-scales",
    "0.9",
    "1.0",
    "1.1",
    "--output-dir",
    str(test_dir),
]

print()
print("======================================")
print("RUNNING 10-SAMPLE MIRA TEST")
print("======================================")
print()

proc = subprocess.Popen(
    cmd,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in proc.stdout:
    print(line, end="")

proc.wait()

wavs = sorted(
    test_dir.glob("*.wav")
)

print()
print("Exit code:", proc.returncode)
print("TEST WAV FILES:", len(wavs))

if proc.returncode != 0:
    raise RuntimeError(
        "Mira 10-sample generation test failed."
    )

if len(wavs) != 10:
    raise RuntimeError(
        f"Expected 10 WAV files, found {len(wavs)}."
    )

print()
print("FIRST:", wavs[0])
print("LAST :", wavs[-1])

print()
print("======================================")
print("MIRA 10-SAMPLE TEST PASSED")
print("======================================")

RuntimeError: Missing model: /content/models/en_US-libritts_r-medium.pt

In [ ]:
# === Mount Drive ===
from google.colab import drive
import os
drive.mount('/content/drive')

DRIVE_DIR = f'/content/drive/MyDrive/{DRIVE_FOLDER}'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive folder: {DRIVE_DIR}')
if MODE == 'bundle':
    BUNDLE_PATH = f'{DRIVE_DIR}/{BUNDLE_NAME}'
    assert os.path.exists(BUNDLE_PATH), (
        f'MODE=bundle but {BUNDLE_PATH} does not exist. Upload your data zip there.')
    print(f'Found bundle: {os.path.getsize(BUNDLE_PATH)/1024/1024:.1f} MB')


In [ ]:
# === Install microWakeWord (kernel-restart-free) ===
# Workarounds for two upstream bugs:
#  1. kahrendt/microWakeWord setup.py has no find_packages() — non-editable
#     install skips the audio/ subpackage. Editable install needs kernel
#     restart, breaks Run All. Fix: install deps + sys.path.insert().
#  2. train.py calls .numpy() on values that newer TF returns as numpy
#     arrays already. Patch with hasattr() guard.
import os, sys, subprocess, importlib, re

DEPS = [
    'audiomentations', 'audio_metadata', 'datasets', 'mmap_ninja', 'numpy',
    'pymicro-features', 'pyyaml', 'tensorflow>=2.16', 'webrtcvad-wheels',
    'ai-edge-litert',
    'git+https://github.com/whatsnowplaying/audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f',
]
print('Installing dependencies...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + DEPS, check=True)

if not os.path.exists('microWakeWord'):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/kahrendt/microWakeWord'], check=True)

MWW_DIR = '/content/microWakeWord'
if MWW_DIR not in sys.path:
    sys.path.insert(0, MWW_DIR)
importlib.invalidate_caches()

fp = '/content/microWakeWord/microwakeword/train.py'
src = open(fp).read()
patched = re.sub(
    r'(\b[a-zA-Z_]+\["[a-z]+"\])\.numpy\(\)',
    r'(\1.numpy() if hasattr(\1, "numpy") else \1)',
    src
)
n = patched.count('hasattr') - src.count('hasattr')
if n > 0:
    open(fp, 'w').write(patched)
    print(f'Patched {n} .numpy() calls in train.py')

import microwakeword
from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration
print('OK: microwakeword.audio.* imports clean')


In [ ]:
# === Data preparation ===
# Mode-aware: either unzip user's bundle, or generate samples inline via Piper.
import os, zipfile

os.chdir('/content')

if MODE == 'bundle':
    print(f'Extracting {BUNDLE_PATH}...')
    with zipfile.ZipFile(BUNDLE_PATH, 'r') as zf:
        zf.extractall('/content')
    for d in ['generated_samples', 'real_recordings', 'confusable_negatives']:
        p = f'/content/{d}'
        if os.path.exists(p):
            n = sum(1 for _ in os.scandir(p) if _.name.endswith('.wav'))
            print(f'  {d}: {n} WAVs')
        else:
            print(f'  {d}: MISSING (will train without)')

elif MODE == 'generate':
    # Defer to the Piper sample-gen cells below
    print('MODE=generate — Piper sample-gen cells will produce samples')

else:
    raise ValueError(f'Unknown MODE: {MODE!r}. Use "bundle" or "generate".')


In [ ]:
# === Piper sample generator install (skipped if MODE=bundle) ===
if MODE == 'generate':
    import glob, os, shutil, subprocess, sys, urllib.request
    PIPER_REPO_DIR = '/content/piper'
    PIPER_SAMPLE_GENERATOR_DIR = '/content/piper-sample-generator'

    subprocess.run(['apt-get', '-qq', 'install', '-y', 'espeak-ng'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
                    'pip', 'setuptools', 'wheel', 'cython'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
                    'piper-tts', 'piper-sample-generator'], check=True)

    if not os.path.exists(PIPER_REPO_DIR):
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/rhasspy/piper', PIPER_REPO_DIR], check=True)
    if not os.path.exists(PIPER_SAMPLE_GENERATOR_DIR):
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/rhasspy/piper-sample-generator',
                        PIPER_SAMPLE_GENERATOR_DIR], check=True)

    PIPER_PYTHON_DIR = f'{PIPER_REPO_DIR}/src/python'
    MA_DIR = f'{PIPER_PYTHON_DIR}/piper_train/vits/monotonic_align'
    MA_IMPORT_DIR = f'{MA_DIR}/monotonic_align'
    MA_BUILD_DIR = f'{MA_DIR}/piper_train/vits/monotonic_align'

    shutil.rmtree(f'{PIPER_PYTHON_DIR}/build', ignore_errors=True)
    shutil.rmtree(MA_IMPORT_DIR, ignore_errors=True)
    shutil.rmtree(f'{MA_DIR}/piper_train', ignore_errors=True)
    os.makedirs(MA_IMPORT_DIR, exist_ok=True)
    os.makedirs(MA_BUILD_DIR, exist_ok=True)
    open(f'{MA_IMPORT_DIR}/__init__.py', 'a').close()
    subprocess.run(f'cd {MA_DIR} && {sys.executable} setup.py build_ext --inplace',
                   shell=True, check=True)
    built = next(iter(glob.glob(f'{MA_BUILD_DIR}/core.*')), None)
    assert built, 'monotonic_align core extension build failed'
    shutil.copy2(built, MA_IMPORT_DIR)

    for path in (PIPER_PYTHON_DIR, PIPER_SAMPLE_GENERATOR_DIR):
        if path not in sys.path:
            sys.path.insert(0, path)

    MODEL_PATH = 'models/en_US-libritts_r-medium.pt'
    MODEL_CONFIG_PATH = f'{MODEL_PATH}.json'
    os.makedirs('models', exist_ok=True)
    if not os.path.exists(MODEL_PATH):
        print('Downloading libritts_r model (~75 MB)...')
        urllib.request.urlretrieve(
            'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt',
            MODEL_PATH)
        urllib.request.urlretrieve(
            'https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/libritts_r/medium/en_US-libritts_r-medium.onnx.json',
            MODEL_CONFIG_PATH)
    print('Piper ready')
else:
    print('Skipped (MODE != generate)')


In [ ]:
# === Generate positive samples (US English, optionally UK) ===
if MODE == 'generate':
    import os, subprocess, sys
    PIPER_SAMPLE_GENERATOR_DIR = '/content/piper-sample-generator'
    MODEL_PATH = 'models/en_US-libritts_r-medium.pt'
    PIPER_BATCH = 256  # drop to 128 on T4 if CUDA OOM

    def run_piper(target_word, max_samples, output_dir):
        cmd = [sys.executable, f'{PIPER_SAMPLE_GENERATOR_DIR}/generate_samples.py',
               target_word, '--phoneme-input', '--model', MODEL_PATH,
               '--max-samples', str(max_samples),
               '--batch-size', str(PIPER_BATCH),
               '--noise-scales', '0.5', '--noise-scale-ws', '0.6',
               '--output-dir', output_dir]
        subprocess.run(cmd, check=True)

    os.makedirs('generated_samples', exist_ok=True)

    print(f'Generating {SAMPLES_US} US samples for {WAKE_WORD_IPA_US!r}...')
    run_piper(WAKE_WORD_IPA_US, SAMPLES_US, 'generated_samples')

    if WAKE_WORD_IPA_UK and SAMPLES_UK > 0:
        print(f'Generating {SAMPLES_UK} UK samples for {WAKE_WORD_IPA_UK!r}...')
        run_piper(WAKE_WORD_IPA_UK, SAMPLES_UK, 'generated_samples')

    n = sum(1 for f in os.listdir('generated_samples') if f.endswith('.wav'))
    print(f'Total positive samples: {n}')
else:
    print('Skipped (MODE != generate)')


In [ ]:
# === Generate confusable negatives ===
if MODE == 'generate':
    import os, subprocess, sys
    from pathlib import Path
    PIPER_SAMPLE_GENERATOR_DIR = '/content/piper-sample-generator'
    MODEL_PATH = 'models/en_US-libritts_r-medium.pt'

    os.makedirs('confusable_negatives', exist_ok=True)
    for phrase in CONFUSABLE_PHRASES:
        safe = phrase.replace(' ', '_').replace(',', '').replace("'", '')
        existing = len(list(Path('confusable_negatives').glob(f'{safe}_*.wav')))
        if existing >= SAMPLES_PER_CONFUSABLE:
            print(f'  {phrase!r}: {existing} already, skip')
            continue
        tmp = f'/tmp/confusable_{safe}'
        os.makedirs(tmp, exist_ok=True)
        print(f'  generating {SAMPLES_PER_CONFUSABLE} for {phrase!r}...')
        subprocess.run([sys.executable, f'{PIPER_SAMPLE_GENERATOR_DIR}/generate_samples.py',
                        phrase, '--model', MODEL_PATH,
                        '--max-samples', str(SAMPLES_PER_CONFUSABLE),
                        '--batch-size', '256',
                        '--noise-scales', '0.5', '--noise-scale-ws', '0.6',
                        '--output-dir', tmp], check=True)
        # Move + rename with safe prefix
        for f in os.listdir(tmp):
            if f.endswith('.wav'):
                os.rename(f'{tmp}/{f}', f'confusable_negatives/{safe}_{f}')

    n = sum(1 for f in os.listdir('confusable_negatives') if f.endswith('.wav'))
    print(f'Total confusable negatives: {n}')
else:
    print('Skipped (MODE != generate)')


In [ ]:
# === Download standard negative datasets (DNS challenge, AudioSet, MIT IRs) ===
# Pre-generated spectrogram features hosted on HuggingFace by kahrendt.
import os, subprocess
from huggingface_hub import snapshot_download

snapshot_download(
    'kahrendt/microwakeword',
    repo_type='dataset',
    local_dir='/content/negative_datasets',
    allow_patterns=['speech/*', 'dinner_party/*', 'no_speech/*',
                    'dinner_party_eval/*'],
)

# MIT room impulse responses for reverb augmentation
if not os.path.exists('mit_rirs') or not os.listdir('mit_rirs'):
    print('Downloading MIT room impulse responses...')
    subprocess.run(['mkdir', '-p', 'mit_rirs'], check=True)
    subprocess.run('cd mit_rirs && wget -q https://www.openslr.org/resources/28/rirs_noises.zip && unzip -q rirs_noises.zip',
                   shell=True, check=True)

# Background noise corpora (FMA + AudioSet 16 kHz subsets)
if not os.path.exists('fma_16k') or not os.listdir('fma_16k'):
    print('Downloading FMA background corpus (~500 MB)...')
    subprocess.run('mkdir -p fma_16k && cd fma_16k && wget -q https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main/fma_16k.tar && tar -xf fma_16k.tar && rm fma_16k.tar',
                   shell=True, check=True)
if not os.path.exists('audioset_16k') or not os.listdir('audioset_16k'):
    print('Downloading AudioSet background corpus (~500 MB)...')
    subprocess.run('mkdir -p audioset_16k && cd audioset_16k && wget -q https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main/audioset_16k.tar && tar -xf audioset_16k.tar && rm audioset_16k.tar',
                   shell=True, check=True)
print('All negative datasets ready')


In [ ]:
# === Augmentation + feature extraction ===
from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration
import os, shutil, traceback
from mmap_ninja.ragged import RaggedMmap

clips = Clips(
    input_directory='generated_samples',
    file_pattern='*.wav',
    max_clip_duration_s=None,
    remove_silence=True,
    random_split_seed=42,
    split_count=0.1,
)
augmenter = Augmentation(
    augmentation_duration_s=3.2,
    augmentation_probabilities={
        'SevenBandParametricEQ': 0.15, 'TanhDistortion': 0.10,
        'PitchShift': 0.15, 'BandStopFilter': 0.10,
        'AddColorNoise': 0.20, 'AddBackgroundNoise': 0.85,
        'Gain': 1.00, 'GainTransition': 0.25, 'RIR': 0.60,
    },
    impulse_paths=['mit_rirs'],
    background_paths=['fma_16k', 'audioset_16k'],
    background_min_snr_db=-5, background_max_snr_db=20,
    min_jitter_s=0.10, max_jitter_s=0.50,
)

os.makedirs('generated_augmented_features', exist_ok=True)
SPLIT_CONFIG = {
    'training':   {'split_name': 'train',      'repetition': 3, 'slide_frames': 10},
    'validation': {'split_name': 'validation', 'repetition': 1, 'slide_frames': 10},
    'testing':    {'split_name': 'test',       'repetition': 1, 'slide_frames': 1 },
}

for split, cfg in SPLIT_CONFIG.items():
    out = f'generated_augmented_features/{split}'
    mmap = f'{out}/wakeword_mmap'
    if os.path.exists(mmap) and list(os.scandir(mmap)):
        print(f'{split}: cached, skipping')
        continue
    if os.path.exists(mmap):
        shutil.rmtree(mmap)
    os.makedirs(out, exist_ok=True)
    print(f'Generating {split} (rep={cfg["repetition"]}, slide={cfg["slide_frames"]})...')
    try:
        sg = SpectrogramGeneration(clips=clips, augmenter=augmenter,
                                    slide_frames=cfg['slide_frames'], step_ms=10)
        RaggedMmap.from_generator(
            out_dir=mmap, batch_size=200, verbose=True,
            sample_generator=sg.spectrogram_generator(
                split=cfg['split_name'], repeat=cfg['repetition']),
        )
    except Exception:
        traceback.print_exc()
        if os.path.exists(mmap): shutil.rmtree(mmap)
        raise
print('Positive features ready')

# Confusable features (only if confusable_negatives/ exists)
if os.path.exists('confusable_negatives') and os.listdir('confusable_negatives'):
    print('Generating confusable features...')
    confusable_clips = Clips(
        input_directory='confusable_negatives', file_pattern='*.wav',
        max_clip_duration_s=None, remove_silence=True,
        random_split_seed=42, split_count=0.1,
    )
    os.makedirs('confusable_features', exist_ok=True)
    for split, cfg in SPLIT_CONFIG.items():
        out = f'confusable_features/{split}'
        mmap = f'{out}/wakeword_mmap'
        if os.path.exists(mmap) and list(os.scandir(mmap)):
            continue
        if os.path.exists(mmap): shutil.rmtree(mmap)
        os.makedirs(out, exist_ok=True)
        sg = SpectrogramGeneration(clips=confusable_clips, augmenter=augmenter,
                                    slide_frames=cfg['slide_frames'], step_ms=10)
        RaggedMmap.from_generator(
            out_dir=mmap, batch_size=200, verbose=True,
            sample_generator=sg.spectrogram_generator(
                split=cfg['split_name'], repeat=cfg['repetition']),
        )
    print('Confusable features ready')

# Real recording features (only if real_recordings/ exists)
if os.path.exists('real_recordings') and os.listdir('real_recordings'):
    print('Generating real-recording features...')
    real_clips = Clips(
        input_directory='real_recordings', file_pattern='*.wav',
        max_clip_duration_s=None, remove_silence=True,
        random_split_seed=42, split_count=0.1,
    )
    os.makedirs('real_recording_features', exist_ok=True)
    for split, cfg in SPLIT_CONFIG.items():
        out = f'real_recording_features/{split}'
        mmap = f'{out}/wakeword_mmap'
        if os.path.exists(mmap) and list(os.scandir(mmap)):
            continue
        if os.path.exists(mmap): shutil.rmtree(mmap)
        os.makedirs(out, exist_ok=True)
        sg = SpectrogramGeneration(clips=real_clips, augmenter=augmenter,
                                    slide_frames=cfg['slide_frames'], step_ms=10)
        RaggedMmap.from_generator(
            out_dir=mmap, batch_size=200, verbose=True,
            sample_generator=sg.spectrogram_generator(
                split=cfg['split_name'], repeat=cfg['repetition']),
        )
    print('Real-recording features ready')


In [ ]:
# === Training config YAML ===
import yaml, os
from pathlib import Path

SKIP_CONFUSABLES = not Path('confusable_features/training/wakeword_mmap').exists()
SKIP_REAL = not Path('real_recording_features/training/wakeword_mmap').exists()

config = {
    'window_step_ms': 10,
    'train_dir': f'trained_models/{OUTPUT_NAME}',
    'features': [
        dict(features_dir='generated_augmented_features', sampling_weight=8.0,
             penalty_weight=2.0, truth=True, truncation_strategy='truncate_start',
             type='mmap'),
        dict(features_dir='negative_datasets/speech', sampling_weight=10.0,
             penalty_weight=2.5, truth=False, truncation_strategy='random', type='mmap'),
        dict(features_dir='negative_datasets/dinner_party', sampling_weight=15.0,
             penalty_weight=3.0, truth=False, truncation_strategy='random', type='mmap'),
        dict(features_dir='negative_datasets/no_speech', sampling_weight=5.0,
             penalty_weight=1.0, truth=False, truncation_strategy='random', type='mmap'),
        dict(features_dir='negative_datasets/dinner_party_eval', sampling_weight=0.0,
             penalty_weight=1.0, truth=False, truncation_strategy='split', type='mmap'),
    ],
    'training_steps': [25000, 20000],
    'positive_class_weight': [2, 2],
    'negative_class_weight': [40, 50],
    'learning_rates': [0.001, 0.0001],
    'batch_size': 256,
    'time_mask_max_size': [5, 5], 'time_mask_count': [1, 1],
    'freq_mask_max_size': [3, 3], 'freq_mask_count': [1, 1],
    'eval_step_interval': 500,
    'clip_duration_ms': 1500,
    'target_minimization': 0.4,
    'minimization_metric': 'ambient_false_positives_per_hour',
    'maximization_metric': 'average_viable_recall',
}

if not SKIP_CONFUSABLES:
    config['features'].append(dict(features_dir='confusable_features', sampling_weight=8.0,
                                    penalty_weight=5.0, truth=False,
                                    truncation_strategy='random', type='mmap'))
if not SKIP_REAL:
    config['features'].append(dict(features_dir='real_recording_features', sampling_weight=8.0,
                                    penalty_weight=2.0, truth=True,
                                    truncation_strategy='truncate_start', type='mmap'))

os.makedirs(f'trained_models/{OUTPUT_NAME}', exist_ok=True)
with open('training_parameters.yaml', 'w') as f:
    yaml.dump(config, f)
print('training_parameters.yaml ready')
print(f'  Feature sets: {len(config["features"])}, total steps: {sum(config["training_steps"])}')


In [ ]:
# === Train the model ===
# subprocess + PYTHONPATH so it can find microwakeword (sys.path doesn't propagate)
import os, sys, subprocess, shutil
shutil.rmtree(f'trained_models/{OUTPUT_NAME}', ignore_errors=True)

env = os.environ.copy()
env['PYTHONPATH'] = '/content/microWakeWord:' + env.get('PYTHONPATH', '')
env['XLA_FLAGS'] = '--xla_gpu_autotune_level=0'

cmd = [
    sys.executable, '-m', 'microwakeword.model_train_eval',
    '--training_config', 'training_parameters.yaml',
    '--train', '1', '--restore_checkpoint', '0',
    '--test_tflite_streaming_quantized', '1',
    '--use_weights', 'best_weights',
    'mixednet',
    '--pointwise_filters', '64,64,64,64',
    '--repeat_in_block', '1, 1, 1, 1',
    '--mixconv_kernel_sizes', '[5], [7,11], [9,15], [23]',
    '--residual_connection', '0,0,0,0',
    '--first_conv_filters', '32',
    '--first_conv_kernel_size', '5',
    '--stride', '3',
]
print('Running:', ' '.join(cmd)); print()
proc = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
proc.wait()
print(); print('Exit code:', proc.returncode)
assert proc.returncode == 0, 'training failed - see output above'


In [ ]:
# === Export + push to Drive ===
import os, json, shutil, datetime

tflite_src = f'trained_models/{OUTPUT_NAME}/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite'
assert os.path.exists(tflite_src), f'No model at {tflite_src}'

OUT_TFLITE = f'{OUTPUT_NAME}.tflite'
OUT_JSON = f'{OUTPUT_NAME}.json'
shutil.copy2(tflite_src, OUT_TFLITE)
print(f'wrote {OUT_TFLITE} ({os.path.getsize(OUT_TFLITE)/1024:.1f} KB)')

manifest = {
    'type': 'micro',
    'wake_word': WAKE_WORD,
    'author': AUTHOR,
    'website': AUTHOR_WEBSITE,
    'model': OUT_TFLITE,
    'trained_languages': TRAINED_LANGUAGES,
    'version': 2,
    'micro': {
        'probability_cutoff': PROBABILITY_CUTOFF,
        'feature_step_size': 10,
        'sliding_window_size': SLIDING_WINDOW_SIZE,
        'tensor_arena_size': TENSOR_ARENA_SIZE,
        'minimum_esphome_version': '2024.7.0',
    }
}
with open(OUT_JSON, 'w') as f:
    json.dump(manifest, f, indent=2)
print(f'wrote {OUT_JSON}')
print(json.dumps(manifest, indent=2))

for fn in (OUT_TFLITE, OUT_JSON):
    shutil.copy2(fn, f'{DRIVE_DIR}/{fn}')
    print(f'pushed {fn} -> {DRIVE_DIR}')

ts = datetime.datetime.now(datetime.timezone.utc).isoformat()
with open(f'{DRIVE_DIR}/_run_finished.txt', 'w') as f:
    f.write(f'Training run finished at {ts}\n')
print()
print(f'DONE. Find your model at /content/drive/MyDrive/{DRIVE_FOLDER}/{OUTPUT_NAME}.tflite')


## Deploying to ESPHome devices

Drop the `.tflite` + `.json` next to your ESPHome YAML (e.g. in `/config/esphome/wakewords/<output_name>/`).

In your device YAML, replace your existing wake word:

```yaml
micro_wake_word:
  models:
    - model: wakewords/<output_name>/<output_name>.json
  on_wake_word_detected:
    - voice_assistant.start:
```

Then `esphome run <device>.yaml` (USB or OTA).

## Manifest tuning (likely needed)

The defaults work for **Hey Harold** specifically. Your model's confidence
distribution will differ. Iterate:

| Symptom | Knob |
|---|---|
| Doesn't fire on the wake word | Lower `probability_cutoff` (try 0.7, 0.6) |
| Fires on too many things | Raise `probability_cutoff` (try 0.92, 0.95) |
| LED fires but no STT response | `sliding_window_size: 5` (faster fire); also check Echo speaker mute |
| `Failed to allocate tensors` log | Raise `tensor_arena_size` (try 50000, 80000) |

## Known gotchas
- Editable install (`pip install -e ./microWakeWord`) requires kernel restart — broken for Run All
- `train.py` upstream calls `.numpy()` on numpy arrays under newer TF (this notebook patches it)
- T4 GPU OOMs during validation — use A100 + High-RAM
- Manifest path mismatch: training writes to `trained_models/<output_name>/`, manifest must match
- `voice_assistant` has no audio lookback; if MWW fires AFTER user speaks, STT gets silence
